# MemoryLane Database Explorer

Interactive notebook for browsing and inspecting the `context_events` table in the MemoryLane SQLite database.

## Setup

In [1]:
%pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.9 MB/s eta 0:00:00m eta 0:00:010:00:01

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3
import os
import pandas as pd
from datetime import datetime, timezone

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)

DB_PATH = os.path.expanduser('~/Library/Application Support/memorylane/memorylane-dev.db')

conn = sqlite3.connect(DB_PATH)
print(f'Connected to {DB_PATH}')
print(f'File size: {os.path.getsize(DB_PATH) / 1024:.1f} KB')

Connected to /Users/filip/Library/Application Support/memorylane/memorylane-dev.db
File size: 1732.0 KB


## Schema

All tables, indexes, triggers, and virtual tables in the database.

In [3]:
schema = pd.read_sql_query(
    "SELECT type, name, sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type, name",
    conn,
)
for _, row in schema.iterrows():
    print(f'-- {row["type"]}: {row["name"]}')
    print(row['sql'])
    print()

-- index: idx_context_events_appName
CREATE INDEX idx_context_events_appName ON context_events(appName)

-- index: idx_context_events_timestamp
CREATE INDEX idx_context_events_timestamp ON context_events(timestamp)

-- table: context_events
CREATE TABLE context_events (
          id TEXT PRIMARY KEY,
          timestamp INTEGER NOT NULL,
          text TEXT NOT NULL DEFAULT '',
          summary TEXT NOT NULL DEFAULT '',
          appName TEXT NOT NULL DEFAULT '',
          vector BLOB
        )

-- table: context_events_fts
CREATE VIRTUAL TABLE context_events_fts USING fts5(
          text,
          summary,
          content='context_events',
          content_rowid='rowid'
        )

-- table: context_events_fts_config
CREATE TABLE 'context_events_fts_config'(k PRIMARY KEY, v) WITHOUT ROWID

-- table: context_events_fts_data
CREATE TABLE 'context_events_fts_data'(id INTEGER PRIMARY KEY, block BLOB)

-- table: context_events_fts_docsize
CREATE TABLE 'context_events_fts_docsize'(id I

## Overview

High-level statistics: total events, date range, capture rate, and events by app.

In [4]:
count = conn.execute('SELECT COUNT(*) FROM context_events').fetchone()[0]
print(f'Total events: {count:,}')

if count > 0:
    oldest, newest = conn.execute(
        'SELECT MIN(timestamp), MAX(timestamp) FROM context_events'
    ).fetchone()
    oldest_dt = datetime.fromtimestamp(oldest / 1000, tz=timezone.utc).astimezone()
    newest_dt = datetime.fromtimestamp(newest / 1000, tz=timezone.utc).astimezone()
    span = newest_dt - oldest_dt
    print(f'Date range: {oldest_dt:%Y-%m-%d %H:%M} \u2192 {newest_dt:%Y-%m-%d %H:%M}')
    print(f'Span: {span}')
    if span.total_seconds() > 0:
        rate = count / (span.total_seconds() / 3600)
        print(f'Capture rate: ~{rate:.1f} events/hour')
    print()
    print('Events by app:')
    apps = pd.read_sql_query(
        'SELECT appName, COUNT(*) as count FROM context_events GROUP BY appName ORDER BY count DESC',
        conn,
    )
    print(apps.to_string(index=False))

Total events: 17
Date range: 2026-02-12 16:25 → 2026-02-13 13:31
Span: 21:05:51.984000
Capture rate: ~0.8 events/hour

Events by app:
      appName  count
       Cursor      9
     Electron      4
                   2
Google Chrome      1
     Terminal      1


## Browse Events

Paginated view of recent events. Adjust `LIMIT` and `OFFSET` to page through.

In [5]:
LIMIT = 20
OFFSET = 0

df = pd.read_sql_query(
    f"""
    SELECT id, timestamp, appName, summary, LENGTH(text) as text_len
    FROM context_events
    ORDER BY timestamp DESC
    LIMIT {LIMIT} OFFSET {OFFSET}
    """,
    conn,
)
df['time'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True).dt.tz_convert('US/Eastern')
df = df[['id', 'time', 'appName', 'summary', 'text_len']]
print(f'Showing {len(df)} events (offset {OFFSET})')
df

Showing 17 events (offset 0)


,id,time,appName,summary,text_len
0,19b2be10-0453-4c28-abd5-2210e562c2a6,2026-02-13 07:31:14.763000-05:00,Cursor,"Managing MemoryLane integrations, monitoring API costs, and storage usage.",286
1,99001a6a-bd9f-46b4-8806-df2862d9c8c9,2026-02-13 07:31:05.754000-05:00,Electron,"Exploring SQLite database with Jupyter notebook, encountering pandas module error.",1542
2,64e151d4-2aaa-41ea-8ae0-70c457cb289e,2026-02-13 07:30:54.371000-05:00,Google Chrome,Editing package.json scripts for MemoryLane application.,4257
3,3530b3c3-0c71-4d71-90df-9ea755b5d7e5,2026-02-13 07:30:52.238000-05:00,Cursor,"Managing MemoryLane integrations, adding to AI assistants, monitoring API costs.",283
4,cb811c35-5f58-4f96-85ba-7f385fffa752,2026-02-13 07:28:23.844000-05:00,,"""Added debug pipeline writer and integration instructions""",3613
5,5f706283-f39f-440d-a860-994b64bc1cc1,2026-02-13 07:28:16.237000-05:00,,"""Completed debug pipeline integration tasks""",2770
6,960e61ce-2dd7-4976-923f-664fa040b0dc,2026-02-13 07:28:11.895000-05:00,Cursor,Managing MemoryLane captures and API integrations before switching to Cursor.,283
7,c9264ca3-1caf-4aff-8180-0890519ebea3,2026-02-12 10:39:51.379000-05:00,Terminal,"```json\n{\n ""summary"": ""Reviewing code changes in index.ts related to OCR and screenshot processing"",\n ""text"": ""...",0
8,22173921-7550-46ac-a2f1-270d61cd7778,2026-02-12 10:39:39.326000-05:00,Electron,Reviewing index.ts changes in memorylane-2 project,1752
9,3ba70d0e-d10f-4d4f-8c99-7d492787eed8,2026-02-12 10:39:30.339000-05:00,Cursor,Reviewing MemoryLane app stats and integrations in Electron,429


## Full-Text Search

Search `text` and `summary` fields using SQLite FTS5. Change `QUERY` to search for different terms.

In [6]:
QUERY = 'memorylane'

fts_df = pd.read_sql_query(
    """
    SELECT ce.id, ce.timestamp, ce.appName, ce.summary, LENGTH(ce.text) as text_len
    FROM context_events_fts fts
    JOIN context_events ce ON ce.rowid = fts.rowid
    WHERE context_events_fts MATCH ?
    ORDER BY ce.timestamp DESC
    """,
    conn,
    params=(QUERY,),
)
fts_df['time'] = pd.to_datetime(fts_df['timestamp'], unit='ms', utc=True).dt.tz_convert('US/Eastern')
fts_df = fts_df[['id', 'time', 'appName', 'summary', 'text_len']]
print(f'Found {len(fts_df)} results for "{QUERY}"')
fts_df

Found 14 results for "memorylane"


,id,time,appName,summary,text_len
0,19b2be10-0453-4c28-abd5-2210e562c2a6,2026-02-13 07:31:14.763000-05:00,Cursor,"Managing MemoryLane integrations, monitoring API costs, and storage usage.",286
1,99001a6a-bd9f-46b4-8806-df2862d9c8c9,2026-02-13 07:31:05.754000-05:00,Electron,"Exploring SQLite database with Jupyter notebook, encountering pandas module error.",1542
2,64e151d4-2aaa-41ea-8ae0-70c457cb289e,2026-02-13 07:30:54.371000-05:00,Google Chrome,Editing package.json scripts for MemoryLane application.,4257
3,3530b3c3-0c71-4d71-90df-9ea755b5d7e5,2026-02-13 07:30:52.238000-05:00,Cursor,"Managing MemoryLane integrations, adding to AI assistants, monitoring API costs.",283
4,cb811c35-5f58-4f96-85ba-7f385fffa752,2026-02-13 07:28:23.844000-05:00,,"""Added debug pipeline writer and integration instructions""",3613
5,5f706283-f39f-440d-a860-994b64bc1cc1,2026-02-13 07:28:16.237000-05:00,,"""Completed debug pipeline integration tasks""",2770
6,960e61ce-2dd7-4976-923f-664fa040b0dc,2026-02-13 07:28:11.895000-05:00,Cursor,Managing MemoryLane captures and API integrations before switching to Cursor.,283
7,22173921-7550-46ac-a2f1-270d61cd7778,2026-02-12 10:39:39.326000-05:00,Electron,Reviewing index.ts changes in memorylane-2 project,1752
8,3ba70d0e-d10f-4d4f-8c99-7d492787eed8,2026-02-12 10:39:30.339000-05:00,Cursor,Reviewing MemoryLane app stats and integrations in Electron,429
9,eed52358-c476-47ef-8e6a-979001305e1d,2026-02-12 10:37:13.234000-05:00,Cursor,Reviewing changes in index.ts and tools.ts in memorylane-2 project,3209


## Filter by App

View events from a specific application. Change `APP_NAME` to filter.

In [7]:
APP_NAME = 'Cursor'

app_df = pd.read_sql_query(
    """
    SELECT id, timestamp, summary, LENGTH(text) as text_len
    FROM context_events
    WHERE appName = ?
    ORDER BY timestamp DESC
    LIMIT 20
    """,
    conn,
    params=(APP_NAME,),
)
app_df['time'] = pd.to_datetime(app_df['timestamp'], unit='ms', utc=True).dt.tz_convert('US/Eastern')
app_df = app_df[['id', 'time', 'summary', 'text_len']]
print(f'Found {len(app_df)} events for app "{APP_NAME}"')
app_df

Found 9 events for app "Cursor"


,id,time,summary,text_len
0,19b2be10-0453-4c28-abd5-2210e562c2a6,2026-02-13 07:31:14.763000-05:00,"Managing MemoryLane integrations, monitoring API costs, and storage usage.",286
1,3530b3c3-0c71-4d71-90df-9ea755b5d7e5,2026-02-13 07:30:52.238000-05:00,"Managing MemoryLane integrations, adding to AI assistants, monitoring API costs.",283
2,960e61ce-2dd7-4976-923f-664fa040b0dc,2026-02-13 07:28:11.895000-05:00,Managing MemoryLane captures and API integrations before switching to Cursor.,283
3,3ba70d0e-d10f-4d4f-8c99-7d492787eed8,2026-02-12 10:39:30.339000-05:00,Reviewing MemoryLane app stats and integrations in Electron,429
4,eed52358-c476-47ef-8e6a-979001305e1d,2026-02-12 10:37:13.234000-05:00,Reviewing changes in index.ts and tools.ts in memorylane-2 project,3209
5,ea8cdb9c-82cc-4357-a33d-fab0d448faa5,2026-02-12 10:37:11.324000-05:00,Extraction failed,0
6,5ae7bfe7-c77f-4ca7-b72e-0953dc15a6b8,2026-02-12 10:25:51.225000-05:00,"Viewed MemoryLane app with 2 screenshots, 4 KB storage, $2.73 API cost",472
7,bd862384-01fc-4ce5-8fee-3a589dcd55a0,2026-02-12 10:25:27.973000-05:00,"Viewing MemoryLane app with 0 screenshots, 4 KB storage, $2.73 API cost",436
8,d7f07d76-6b6c-4325-86a7-be677fbd9231,2026-02-12 10:25:22.779000-05:00,"User stopped capture in MemoryLane app, viewed storage and API cost",442


## Event Detail

Inspect a single event's full text and summary. Paste an event ID from the tables above.

In [8]:
EVENT_ID = df.iloc[0]['id'] if len(df) > 0 else ''

if EVENT_ID:
    row = conn.execute(
        'SELECT id, timestamp, appName, summary, text FROM context_events WHERE id = ?',
        (EVENT_ID,),
    ).fetchone()
    if row:
        event_id, ts, app, summary, text = row
        event_time = datetime.fromtimestamp(ts / 1000, tz=timezone.utc).astimezone()
        print(f'ID:      {event_id}')
        print(f'Time:    {event_time:%Y-%m-%d %H:%M:%S}')
        print(f'App:     {app}')
        print(f'Summary: {summary}')
        print(f'\n--- Full Text ({len(text):,} chars) ---\n')
        print(text)
    else:
        print(f'No event found with ID: {EVENT_ID}')
else:
    print('No events to display. Run the Browse Events cell first.')

ID:      19b2be10-0453-4c28-abd5-2210e562c2a6
Time:    2026-02-13 13:31:14
App:     Cursor
Summary: Managing MemoryLane integrations, monitoring API costs, and storage usage.

--- Full Text (286 chars) ---

Finder
File
Edit
View
Go
Window
Help
T1 lu0
US
Q
8
Fri 13. 2.
13:31
• MemoryLane
MemoryLane
• Stop Capture
15
Screenshots
1.7 MB
Storage
$2.93
API Cost
Integrations
Register MemoryLane as an MCP
server for AI assistants.
Add to Claude
Add
to
Cursor
Claude
Code
sk-or-v...a73d
Manage Key


## Cleanup

In [9]:
conn.close()
print('Connection closed.')

Connection closed.
